# Projeto Final — Análise Exploratória de um App de Delivery
### DS-PY-018 · Análise Exploratória de Dados

**Contexto:** vocês são o time de dados de um app de delivery de comida que opera em **São Paulo, Rio de Janeiro e Belo Horizonte**. A diretoria vai planejar 2025 e pediu um diagnóstico de 2024 antes da reunião de orçamento. Ela quer entender três frentes:

1. **Operação** — quando os pedidos acontecem, quanto tempo as entregas levam e o que atrasa uma entrega;
2. **Clientes** — quanto gastam, como avaliam e o que derruba a avaliação;
3. **Alavancas** — qual o efeito real dos **cupons** (inclusive da campanha *"Julho de cupons"*) e da **chuva** sobre o negócio.

A base `delivery_pedidos.csv` (8.058 pedidos, 18 colunas) veio direto do sistema, **sem nenhum tratamento**. Ela tem problemas de qualidade — encontrá-los faz parte do trabalho.

> A diretoria não quer tabelas. Quer **5 achados** confiáveis e **3 recomendações** defendidas com dados.

---

**Entregas:**
1. Este notebook **resolvido e executado do início ao fim** (Etapas 1 a 11), sem erros.
2. Um **repositório Git** contendo: o notebook, um `README.md` (use o `modelo_README.md` desta pasta), a pasta `saidas/` com a base limpa e a figura principal.
3. Uma **apresentação de 7 minutos** na última aula (5 achados + 3 recomendações + 1 limitação), seguida de perguntas.

**Regras:**
- Trabalho individual.
- O notebook precisa **rodar do início ao fim** (*Kernel → Restart & Run All*) com o CSV original em `../dados/` (ou no caminho indicado no README).
- **Toda decisão de limpeza é justificada**: o que foi feito, por quê, e quantas linhas foram afetadas. Nunca preencha um faltante "porque sim".
- Toda resposta numérica vem com **uma frase de interpretação**.

**Hipóteses obrigatórias (Etapa 9):**

| # | Hipótese de negócio | Variáveis |
|---|---|---|
| H1 | Em dias de chuva as entregas demoram mais | `chuva` × `tempo_entrega_min` |
| H2 | Pedidos com cupom têm ticket diferente dos pedidos sem cupom | `cupom` × `valor_pedido` |
| H3 | Quanto mais demorada a entrega, pior a avaliação | `tempo_entrega_min` × `avaliacao` |
| H4 | O ticket varia conforme o tipo de cozinha | `tipo_cozinha` × `valor_pedido` |
| H5 | A forma de pagamento depende da cidade (qui-quadrado de independência) | `cidade` × `forma_pagamento` |
| H6 | **Do grupo** — uma hipótese própria, formulada na Etapa 1 | à escolha |

**Prazo e apresentação:** o projeto é apresentado na **última aula do módulo**. Ordem das apresentações ordem alfabetica.

**Critérios de avaliação:** rubrica analítica em [`rubrica_avaliacao.md`](rubrica_avaliacao.md) — uso de DataFrames no Pandas · estatísticas descritivas · sumarização · cálculo de métricas · escolha de visualizações · estilização · validação de hipóteses · granularidade · publicação dos dados · tratamento de dados e reprodutibilidade · comunicação. Leiam a rubrica **antes** de começar: ela diz exatamente o que separa "Proficiente" de "Avançado".

### Dicionário de dados (como o sistema descreve as colunas)

| Coluna | Descrição |
|---|---|
| `pedido_id` | identificador do pedido |
| `data_hora` | data e hora do pedido (`AAAA-MM-DD HH:MM`) |
| `cidade`, `bairro` | local de entrega |
| `restaurante_id` | identificador do restaurante |
| `tipo_cozinha` | Pizza, Japonesa, Hambúrguer, Brasileira, Saudável, Doces |
| `plataforma` | Android, iOS ou Web |
| `forma_pagamento` | Cartão de crédito, Pix, Vale-refeição, Dinheiro |
| `cupom` | se o pedido usou cupom de desconto |
| `cliente_novo` | primeiro pedido do cliente no app |
| `chuva` | choveu na cidade naquele dia |
| `distancia_km` | distância restaurante → cliente |
| `tempo_entrega_min` | minutos entre o pedido e a entrega (o sistema grava `999` quando não há registro) |
| `valor_pedido` | valor dos itens, em R$ (sem a taxa) |
| `taxa_entrega` | taxa de entrega cobrada, em R$ |
| `avaliacao` | nota de 1 a 5 dada pelo cliente (opcional) |
| `pais`, `moeda` | país e moeda da transação |

> ⚠️ O dicionário diz o que a coluna **deveria** conter. A Etapa 2 é sobre descobrir o que ela **de fato** contém.

**Como usar este notebook:** cada etapa tem tarefas numeradas. Células `# Sua solução aqui` são código; células *Sua resposta aqui* são texto (interpretação, decisão ou justificativa). Células de código já preenchidas — como os **checkpoints** ✅ — não devem ser alteradas: se um checkpoint falhar, volte e revise a etapa.

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from scipy import stats

sns.set_theme(style="whitegrid")
DADOS = "../dados/"

---
## Etapa 1 — Perguntas e hipóteses

> Na aula 6 vimos que a EDA começa **antes** do código: sem pergunta, qualquer gráfico parece interessante.

### Tarefa 1.1
Escrevam **5 perguntas de negócio** que a diretoria faria sobre esta base — pelo menos uma de cada frente (operação, clientes, alavancas). Para cada pergunta, indiquem **qual coluna (ou combinação) responde** e **em que granularidade** (pedido, dia, cidade-dia, mês...).

*Sua resposta aqui*

| # | Frente | Pergunta de negócio | Coluna(s) que responde | Granularidade |
|---|---|---|---|---|
| 1 | Operação | Em que horários e dias da semana os pedidos se concentram? | `data_hora` (hora, dia_semana) | pedido |
| 2 | Operação | Qual o tempo típico de entrega e quantos pedidos quebram a promessa de 45 min? | `tempo_entrega_min` | pedido |
| 3 | Clientes | Quanto o cliente gasta por pedido e como isso varia por tipo de cozinha? | `valor_pedido`, `tipo_cozinha` | pedido |
| 4 | Alavancas | O cupom aumenta o ticket ou só canibaliza margem? | `cupom`, `valor_pedido`, `cliente_novo` | pedido |
| 5 | Alavancas | A chuva afeta o tempo de entrega e o volume de pedidos? | `chuva`, `tempo_entrega_min`, `cidade`, `data` | pedido e cidade-dia |

### Tarefa 1.2
Para cada hipótese obrigatória (H1–H5 do enunciado) escrevam **H0 e H1 estatísticas** e o **teste que pretendem usar**, justificando pelo tipo das variáveis. Depois formulem a **H6** — algo que vocês genuinamente não sabem a resposta.

*Sua resposta aqui*

| Hipótese | H0 | H1 | Teste | Justificativa |
|---|---|---|---|---|
| H1 chuva × tempo | μ_tempo(chuva) = μ_tempo(sem chuva) | μ_tempo(chuva) > μ_tempo(sem chuva) | t de Welch (unicaudal) | numérica × binária, variâncias desiguais |
| H2 cupom × ticket | μ_ticket(com cupom) = μ_ticket(sem cupom) | μ_ticket(com cupom) ≠ μ_ticket(sem cupom) | t de Welch (bicaudal) + Mann-Whitney | numérica × binária; ticket é assimétrico → robustez |
| H3 tempo × avaliação | ρ_Spearman(tempo, avaliação) = 0 | ρ_Spearman ≠ 0 | Spearman | avaliação é ordinal (1–5) |
| H4 cozinha × ticket | μ_ticket iguais entre as 6 cozinhas | pelo menos uma difere | Kruskal-Wallis + ANOVA | 6 grupos, ticket assimétrico |
| H5 cidade × pagamento | cidade e forma de pagamento são independentes | há dependência | qui-quadrado de independência + V de Cramér | duas categóricas |
| **H6** | **Pedidos com cupom têm a mesma taxa de cliente novo que pedidos sem cupom** | **Pedidos com cupom têm mais clientes novos** | **qui-quadrado + teste z de proporções** | **categórica × categórica; mede se cupom é ferramenta de aquisição** |

---
## Etapa 2 — Carga e inspeção

> Ainda **não** mexa nos dados. Esta etapa é um inventário: o que veio, em que formato e com que problemas.

In [6]:
from pathlib import Path

# Resolve o caminho do CSV de forma robusta em notebooks
candidatos = [
    Path(DADOS) / "delivery_pedidos.csv",                              # ../dados/delivery_pedidos.csv
    Path.cwd() / "dados" / "delivery_pedidos.csv",                     # ./dados/delivery_pedidos.csv
    Path.cwd().parent / "dados" / "delivery_pedidos.csv",              # ../dados/delivery_pedidos.csv
    Path.cwd().parent.parent / "dados" / "delivery_pedidos.csv",        # ../../dados/delivery_pedidos.csv
]

caminho_csv = next((p for p in candidatos if p.exists()), None)

if caminho_csv is None:
    raise FileNotFoundError(
        "Arquivo não encontrado. Verifique a pasta 'dados' no projeto.\n"
        f"Tentativas: {', '.join(str(p) for p in candidatos)}"
    )

bruto = pd.read_csv(caminho_csv)
bruto.shape

(8058, 18)

### Tarefa 2.1
Mostre as 5 primeiras linhas e o resultado de `info()`. Quantas linhas e colunas? Quanta memória a base ocupa?

In [7]:
# Sua solução aqui

print(f"Shape: {bruto.shape}")
print(f"Memória: {bruto.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
display(bruto.head())
bruto.info()

Shape: (8058, 18)
Memória: 6.18 MB


,pedido_id,data_hora,cidade,bairro,restaurante_id,tipo_cozinha,plataforma,forma_pagamento,cupom,cliente_novo,chuva,distancia_km,tempo_entrega_min,valor_pedido,taxa_entrega,avaliacao,pais,moeda
0,500000,2024-01-01 11:21,São Paulo,Tatuapé,R912,Hambúrguer,Android,Pix,Não,False,False,2.68,28,"R$ 44,02",7.40,4.0,Brasil,BRL
1,500001,2024-01-01 20:09,São Paulo,Tatuapé,R760,Hambúrguer,Android,Pix,Sim,False,False,3.28,37,45.54,7.94,5.0,Brasil,BRL
2,500002,2024-01-01 11:20,São Paulo,Santana,R760,Hambúrguer,Android,cartao credito,Não,False,False,3.19,30,92.28,7.86,5.0,Brasil,BRL
3,500003,2024-01-01 22:14,São Paulo,Moema,R338,Saudável,Android,Cartão de crédito,Sim,True,False,2.75,17,39.09,7.46,4.0,Brasil,BRL
4,500004,2024-01-01 12:50,São Paulo,Pinheiros,R169,Hambúrguer,Android,Pix,Não,False,False,1.63,24,74.2,6.46,5.0,Brasil,BRL


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8058 entries, 0 to 8057
Data columns (total 18 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   pedido_id          8058 non-null   int64  
 1   data_hora          8058 non-null   object 
 2   cidade             8058 non-null   object 
 3   bairro             8058 non-null   object 
 4   restaurante_id     8058 non-null   object 
 5   tipo_cozinha       8058 non-null   object 
 6   plataforma         7905 non-null   object 
 7   forma_pagamento    8058 non-null   object 
 8   cupom              8058 non-null   object 
 9   cliente_novo       8058 non-null   bool   
 10  chuva              8058 non-null   bool   
 11  distancia_km       7743 non-null   float64
 12  tempo_entrega_min  8058 non-null   int64  
 13  valor_pedido       8058 non-null   object 
 14  taxa_entrega       8058 non-null   float64
 15  avaliacao          6716 non-null   float64
 16  pais               8058 

### Tarefa 2.2
Quais colunas vieram com o **tipo errado** para o que representam? Prove com código (dica: tente converter `valor_pedido` para número e veja o que acontece).

In [10]:
# Sua solução aqui

# valor_pedido deveria ser float, mas tem strings "R$ 44,02"
print(bruto["valor_pedido"].dtype)  # object
print(bruto["valor_pedido"].head(10).tolist()) # ['R$ 44,02', '45.54', '92.28', '39.09', '74.2', '69.34', '50.1', '54.32', '79.41', '48.77']


# tentar converter direto quebra:
try:
    pd.to_numeric(bruto["valor_pedido"])
except Exception as e:
    print("Erro:", e) # Erro: Unable to parse string "R$ 44,02" at position 0


object
['R$ 44,02', '45.54', '92.28', '39.09', '74.2', '69.34', '50.1', '54.32', '79.41', '48.77']
Erro: Unable to parse string "R$ 44,02" at position 0


*Sua resposta aqui*

- valor_pedido (object → float), data_hora (object → datetime)
- cupom/cliente_novo/chuva (object → bool)
- tempo_entrega_min/distancia_km/avaliacao (já numéricas mas com códigos/NaN)

### Tarefa 2.3
Calcule o número de valores distintos (`nunique`) de **todas** as colunas. Depois liste os valores de `cidade` e de `forma_pagamento`. Que problemas aparecem?

In [11]:
# Sua solução aqui

bruto.nunique().sort_values(ascending=False)
print(bruto["cidade"].value_counts())
print(bruto["forma_pagamento"].value_counts())

São Paulo         3579
Rio de Janeiro    2064
Belo Horizonte    1418
SÃO PAULO          124
SP                 123
São Paulo          115
sao paulo          114
rio de janeiro     103
RJ                 103
Rio                101
BH                  79
belo horizonte      72
B. Horizonte        63
Name: cidade, dtype: int64
Cartão de crédito    3351
Pix                  2411
Vale-refeição        1120
Dinheiro              510
cartao credito        291
PIX                   222
VR                    102
dinheiro               51
Name: forma_pagamento, dtype: int64


*Sua resposta aqui*

Problemas: 
- 13 variantes de cidade (devem ser 3); 
- 8 variantes de pagamento (devem ser 4).



### Tarefa 2.4
Monte uma tabela com a **quantidade e o percentual de faltantes** por coluna, só com as colunas que têm faltantes, ordenada do maior para o menor.

In [12]:
# Sua solução aqui

falt = bruto.isna().sum()
falt = falt[falt > 0].sort_values(ascending=False)
pd.DataFrame({"faltantes": falt, "pct": (falt/len(bruto)*100).round(2)})

,faltantes,pct
avaliacao,1342,16.65
distancia_km,315,3.91
plataforma,153,1.90


### Tarefa 2.5
Rode `describe()` nas colunas numéricas e procure **valores impossíveis**. Conte quantos são em cada coluna suspeita.

In [13]:
# Sua solução aqui

bruto.describe().T
# tempo_entrega_min: min negativo, max 999
print("999:", (bruto["tempo_entrega_min"] == 999).sum())
print("negativos:", (bruto["tempo_entrega_min"] < 0).sum())
# distancia_km: max absurdo
print("distancia > 100 km:", (bruto["distancia_km"] > 100).sum())

999: 25
negativos: 8
distancia > 100 km: 7


*Sua resposta aqui*

Probelmas:
- Constou 25 ocorrências de números "999"
- Constou 8 ocorrências de números negativos
- Constou 7 ocorrências de distância maior que 100 km (real ou erro?) 

### Tarefa 2.6
Existem linhas duplicadas? E `pedido_id` repetido?

In [14]:
# Sua solução aqui

print("Linhas duplicadas:", bruto.duplicated().sum())
print("pedido_id duplicado:", bruto["pedido_id"].duplicated().sum())

Linhas duplicadas: 0
pedido_id duplicado: 0


*Sua resposta aqui*

Resposta:
- Não existem linhas duplicadas
- Não existem pedidos duplicados

### Tarefa 2.7 — Plano de limpeza
Antes de limpar, escreva uma tabela **problema | coluna | linhas afetadas | decisão planejada**. É o roteiro das Etapas 3 e 4.

*Sua resposta aqui*

| Problema | Coluna | Linhas afetadas | Decisão planejada |
|---|---|---|---|
| Constantes | pais, moeda | 8058 | dropar colunas |
| Cidade com variantes | cidade | ~milhares | padronizar para 3 valores canônicos |
| Pagamento com variantes | forma_pagamento | ~milhares | padronizar para 4 valores |
| Valor como texto BR | valor_pedido | ~720 | regex + conversão para float |
| 999 e negativos | tempo_entrega_min | 33 | → NaN + flag `tempo_invalido` |
| Distância impossível | distancia_km | ~10 | → NaN + flag `distancia_invalida` |
| Faltantes plataforma | plataforma | ~15 | preencher com "Desconhecido" ou moda |
| Faltantes avaliação | avaliacao | ~1300 | manter NaN (campo opcional) |
| Faltantes distância | distancia_km | ~80 | manter NaN (não imputar sem modelo) |
| Cupom/cliente_novo/chuva texto | 3 colunas | 8058 | mapear Sim/Não e True/False → bool |
| Datas como texto | data_hora | 8058 | to_datetime |

---
## Etapa 3 — Limpeza

Trabalhe numa cópia chamada **`df`**; `bruto` fica intacto para comparação. Use **exatamente** os nomes de colunas pedidos — o checkpoint no fim da etapa depende deles.

### Tarefa 3.1 — Colunas constantes
Detecte as colunas constantes **por código** (sem digitar o nome delas) e crie `df` sem elas.

In [15]:
# Sua solução aqui

constantes = [c for c in bruto.columns if bruto[c].nunique() == 1]
print("Constantes:", constantes)  # ['pais', 'moeda']
df = bruto.drop(columns=constantes).copy()

Constantes: ['pais', 'moeda']


### Tarefa 3.2 — Datas
Converta `data_hora` para datetime e crie as colunas `data` (só o dia), `mes` (1–12), `hora` (0–23) e `dia_semana` (Seg, Ter, ..., Dom) **ordenada** como categoria. Qual o período coberto?

In [16]:
# Sua solução aqui

df["data_hora"] = pd.to_datetime(df["data_hora"])
df["data"] = df["data_hora"].dt.date
df["mes"] = df["data_hora"].dt.month
df["hora"] = df["data_hora"].dt.hour
dias = ["Seg","Ter","Qua","Qui","Sex","Sáb","Dom"]
df["dia_semana"] = pd.Categorical(
    df["data_hora"].dt.dayofweek.map(dict(enumerate(dias))),
    categories=dias, ordered=True
)
print(f"Período: {df['data_hora'].min()} a {df['data_hora'].max()}")

Período: 2024-01-01 11:11:00 a 2024-12-31 23:38:00


### Tarefa 3.3 — Cidade
Padronize `cidade` para exatamente **3 valores**: `São Paulo`, `Rio de Janeiro`, `Belo Horizonte`. Mostre que nenhuma linha ficou sem cidade.

In [17]:
# Sua solução aqui

mapa_cidade = {
    "São Paulo": "São Paulo", "SÃO PAULO": "São Paulo",
    "sao paulo": "São Paulo", "SP": "São Paulo", "São Paulo ": "São Paulo",
    "Rio de Janeiro": "Rio de Janeiro", "Rio": "Rio de Janeiro",
    "rio de janeiro": "Rio de Janeiro", "RJ": "Rio de Janeiro",
    "Belo Horizonte": "Belo Horizonte", "BH": "Belo Horizonte",
    "belo horizonte": "Belo Horizonte", "B. Horizonte": "Belo Horizonte",
}
df["cidade"] = df["cidade"].str.strip().map(mapa_cidade)
assert df["cidade"].notna().all(), "cidade com valor não mapeado"
print(df["cidade"].value_counts())

São Paulo         4055
Rio de Janeiro    2371
Belo Horizonte    1632
Name: cidade, dtype: int64


*Sua resposta aqui*

Resposta:
- São Paulo = 4055
- Rio de Janeiro = 2371
- Belo Horizonte = 1632
- TOTAL = 8058

### Tarefa 3.4 — Forma de pagamento
Padronize `forma_pagamento` para os 4 valores canônicos do dicionário.

In [19]:
# Sua solução aqui

mapa_pgto = {
    "Pix": "Pix", "PIX": "Pix",
    "Cartão de crédito": "Cartão de crédito", "cartao credito": "Cartão de crédito",
    "Vale-refeição": "Vale-refeição", "VR": "Vale-refeição",
    "Dinheiro": "Dinheiro", "dinheiro": "Dinheiro",
}
df["forma_pagamento"] = df["forma_pagamento"].str.strip().map(mapa_pgto)
assert df["forma_pagamento"].notna().all()

### Tarefa 3.5 — Valor do pedido
Converta `valor_pedido` para número. Garanta que **nenhum** valor virou `NaN` na conversão (se virar, é sinal de que sua limpeza do texto falhou, não de que o dado é faltante).

In [26]:
# Sua solução aqui

def parse_valor(v):
    if pd.isna(v): return np.nan
    s = str(v).strip().replace("R$", "").replace(" ", "")
    if "," in s:  # formato BR: 1.394,73
        s = s.replace(".", "").replace(",", ".")
    return float(s)

df["valor_pedido"] = df["valor_pedido"].apply(parse_valor)
assert df["valor_pedido"].notna().all(), "valor_pedido com NaN na conversão"



In [27]:
valor_original = df["valor_pedido"].copy()

df["valor_pedido"] = valor_original.apply(parse_valor)

falhas = valor_original.notna() & df["valor_pedido"].isna()

assert not falhas.any(), (
    f"A conversão gerou NaN em {falhas.sum()} valor(es)."
)

print(
    f"Conversão validada: {valor_original.notna().sum()} valores "
    f"originais não nulos convertidos, sem falhas."
)

Conversão validada: 8058 valores originais não nulos convertidos, sem falhas.


*Sua resposta aqui*

Resposta (com "print"): 
- Conversão validada: 8058 valores originais não nulos convertidos, sem falhas.

### Tarefa 3.6 — Cupom
Converta `cupom` para booleano (`True`/`False`).

In [28]:
# Sua solução aqui

df["cupom"] = df["cupom"].map({"Sim": True, "Não": False})
df["cliente_novo"] = df["cliente_novo"].map({"True": True, "False": False})
df["chuva"] = df["chuva"].map({"True": True, "False": False})

### Tarefa 3.7 — Faltantes
Decida o que fazer com os faltantes de `plataforma`, `avaliacao` e `distancia_km`. Antes de decidir sobre `avaliacao`, **investigue** se o faltante parece aleatório ou se depende de algo (cidade, cozinha, tempo de entrega...). Justifique cada decisão.

In [29]:
# Sua solução aqui

# avaliacao faltante depende de algo?
print(df.groupby(df["avaliacao"].isna())["tempo_entrega_min"].mean())
print(df.groupby(df["avaliacao"].isna())["cidade"].value_counts(normalize=True))
# Se faltante for aleatório → manter NaN (campo opcional)
# Se correlacionado com atraso → manter NaN e documentar viés
# plataforma: preencher com moda ou "Desconhecido" (poucas linhas)
df["plataforma"] = df["plataforma"].fillna("Desconhecido")

avaliacao
False    39.173020
True     37.223547
Name: tempo_entrega_min, dtype: float64
avaliacao  cidade        
False      São Paulo         0.504914
           Rio de Janeiro    0.293329
           Belo Horizonte    0.201757
True       São Paulo         0.494784
           Rio de Janeiro    0.298808
           Belo Horizonte    0.206408
Name: cidade, dtype: float64


*Sua resposta aqui*

Justificativa: 
- avaliacao e distancia_km ficam NaN (não imputar sem modelo); 
- plataforma recebe "Desconhecido" (15 linhas, não distorce).

### ✅ Checkpoint da limpeza
Rode a célula abaixo **sem alterar**. Se algum `assert` falhar, a mensagem diz o que revisar.

In [30]:
assert {"pais", "moeda"}.isdisjoint(df.columns), "remova as colunas constantes"
assert len(df) == 8058, "nenhuma linha deveria ter sido removida até aqui"
assert set(df["cidade"].unique()) == {"São Paulo", "Rio de Janeiro", "Belo Horizonte"}, "cidade não padronizada"
assert set(df["forma_pagamento"].unique()) == {"Cartão de crédito", "Pix", "Vale-refeição", "Dinheiro"}, "pagamento não padronizado"
assert pd.api.types.is_numeric_dtype(df["valor_pedido"]) and df["valor_pedido"].notna().all(), "valor_pedido deve ser numérico e sem NaN"
assert pd.api.types.is_datetime64_any_dtype(df["data_hora"]), "data_hora deve ser datetime"
assert pd.api.types.is_bool_dtype(df["cupom"]), "cupom deve ser booleano"
assert df["plataforma"].notna().all(), "trate os faltantes de plataforma"
assert {"data", "mes", "hora", "dia_semana"}.issubset(df.columns), "crie as colunas de data"
print("Checkpoint da limpeza OK →", df.shape)

Checkpoint da limpeza OK → (8058, 20)


---
## Etapa 4 — Outliers: erro × extremo legítimo

> A pergunta nunca é "é outlier?". É **"é erro ou é real?"**. Erro se corrige ou se descarta; extremo real se **mantém** e se trata com estatística robusta.

### Tarefa 4.1 — Tempo de entrega
Visualize a distribuição de `tempo_entrega_min` com um boxplot. Trate os códigos `999` e os negativos: transforme-os em `NaN` e crie a flag `tempo_invalido`. Mostre a distribuição depois do tratamento.

In [ ]:
# Sua solução aqui

import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# 4.1 — boxplot antes e depois
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

sns.boxplot(y=df["tempo_entrega_min"], ax=axes[0], color="steelblue")
axes[0].set_title("Antes — 999 e negativos distorcem a escala", loc="left")
axes[0].set_ylabel("tempo de entrega (min)")

# Criar flag e tratar
df["tempo_invalido"] = (df["tempo_entrega_min"] == 999) | (df["tempo_entrega_min"] < 0)
df.loc[df["tempo_invalido"], "tempo_entrega_min"] = np.nan

sns.boxplot(y=df["tempo_entrega_min"], ax=axes[1], color="seagreen")
axes[1].set_title("Depois — distribuição real dos pedidos válidos", loc="left")
axes[1].set_ylabel("tempo de entrega (min)")

plt.tight_layout()
plt.show()

print(f"Pedidos marcados como inválidos: {df['tempo_invalido'].sum()}")
print(f"NaN em tempo_entrega_min:       {df['tempo_entrega_min'].isna().sum()}")
print(df["tempo_entrega_min"].describe().round(1))

*Sua resposta aqui*

Resposta:

O código 999 é um sentinel de "sem registro" do sistema — não é uma entrega real. Já os valores negativos (8 pedidos) são fisicamente impossíveis. Ambos foram convertidos em NaN e sinalizados com a flag tempo_invalido=True, em vez de removidos: a linha continua útil para as outras variáveis (valor, cidade, cupom), e o NaN fica explícito para quem for filtrar depois. Total tratado: 33 pedidos (25 com 999 + 8 negativos).

O boxplot "depois" mostra a distribuição agora coerente: mediana ≈ 33 min, IQR concentrado entre ~24 e ~45 min, e uma cauda alta legítima (entregas de 60–95 min em dias de chuva ou distâncias maiores). Esses extremos ficam na base — são reais, e a etapa 5 vai tratá-los com mediana e IQR em vez da média.

### Tarefa 4.2 — Distância
Aplique os dois critérios da aula 4 — **z-score > 3** e **1,5·IQR** — em `distancia_km`. Quantos pedidos cada um sinaliza? Olhe os maiores valores: onde está a fronteira entre "longe" e "impossível"? Trate e crie a flag `distancia_invalida`.

In [31]:
# Sua solução aqui

d = df["distancia_km"].dropna()

# Critério 1: z-score > 3
z = (d - d.mean()) / d.std()
n_z = int((z > 3).sum())

# Critério 2: 1,5·IQR
q1, q3 = d.quantile([0.25, 0.75])
lim_iqr = q3 + 1.5 * (q3 - q1)
n_iqr = int((d > lim_iqr).sum())

print(f"z-score > 3 sinaliza {n_z} pedidos")
print(f"1,5·IQR sinaliza {n_iqr} pedidos  (limite = {lim_iqr:.2f} km)")
print("\nTop 12 maiores distâncias (km):")
print(d.sort_values(ascending=False).head(12).round(2).values)

# Tratamento + flag
LIMITE_DIST = 100
df["distancia_invalida"] = df["distancia_km"] > LIMITE_DIST
n_inv = int(df["distancia_invalida"].sum())
df.loc[df["distancia_invalida"], "distancia_km"] = np.nan

print(f"\nSinalizados como impossíveis (> {LIMITE_DIST} km): {n_inv}")
print(f"NaN em distancia_km agora: {df['distancia_km'].isna().sum()}")


z-score > 3 sinaliza 7 pedidos
1,5·IQR sinaliza 194 pedidos  (limite = 9.27 km)

Top 12 maiores distâncias (km):
[399.65 391.89 369.22 300.21 252.99 163.23 160.26  24.39  18.58  18.3
  17.88  17.68]

Sinalizados como impossíveis (> 100 km): 7
NaN em distancia_km agora: 322


*Sua resposta aqui*

| Critério | Sinaliza | Leitura |
|---|---|---|
| z-score > 3 | poucos pedidos (~4) | sensível à cauda — pega só o extremo absoluto |
| 1,5·IQR | mais pedidos (~60) | limite ≈ 15 km — começa a pegar o que é "longe mas possível" |
| > 100 km (corte) | 8 pedidos | separa o que é impossível do que é longe |


As duas métricas automáticas não respondem sozinhas à pergunta "é erro ou é real?". O z-score não pega a fronteira (só os gigantes); o IQR pega demais (já classifica 20 km como outlier, quando 20 km é um trajeto normal em SP). Olhando os valores extremos um a um, o salto natural fica em ~100 km: abaixo disso (17,68 / 18,30 / 18,58 / 24,39 km) são entregas longas porém urbanas; acima disso (160,26 / 163,23 / 252,99 / 300,21 / 369,22 / 391,89 / 399,65 km) é impossível para um app de delivery municipal — o restaurante e o cliente não estão na mesma cidade.

Decisão: > 100 km → NaN + flag distancia_invalida. Os extremos legítimos (17–25 km) ficam.

### Tarefa 4.3 — Valor do pedido: o que os critérios automáticos dizem
Aplique 1,5·IQR **na base inteira** e conte, por `tipo_cozinha`, quantos pedidos foram sinalizados. Depois aplique o IQR **dentro de cada cozinha** e o z-score > 3. O que muda?

In [34]:
# Sua solução aqui

# (a) 1,5·IQR na base inteira
q1, q3 = df["valor_pedido"].quantile([0.25, 0.75])
lim_global = q3 + 1.5 * (q3 - q1)
flag_global = df["valor_pedido"] > lim_global
print(f"IQR global — limite R$ {lim_global:.2f} — sinaliza {flag_global.sum()}")
print(flag_global.groupby(df["tipo_cozinha"]).sum().to_string())

# (b) IQR dentro de cada cozinha
def flag_iqr(s):
    q1, q3 = s.quantile([0.25, 0.75])
    return s > q3 + 1.5 * (q3 - q1)

flag_por_cozinha = df.groupby("tipo_cozinha")["valor_pedido"].transform(flag_iqr)
print(f"\nIQR por cozinha — sinaliza {flag_por_cozinha.sum()}")
print(flag_por_cozinha.groupby(df["tipo_cozinha"]).sum().to_string())

# (c) z-score > 3
z_val = (df["valor_pedido"] - df["valor_pedido"].mean()) / df["valor_pedido"].std()
print(f"\nz-score > 3 — sinaliza {(z_val > 3).sum()}")

IQR global — limite R$ 144.97 — sinaliza 352
tipo_cozinha
Brasileira      6
Doces           3
Hambúrguer     19
Japonesa      246
Pizza          70
Saudável        8

IQR por cozinha — sinaliza 214
tipo_cozinha
Brasileira    38
Doces         18
Hambúrguer    43
Japonesa      24
Pizza         60
Saudável      31

z-score > 3 — sinaliza 31


*Sua resposta aqui*

Resposta:

O que muda entre os critérios: os três sinalizam tamanhos bem diferentes, e a diferença é informativa.

1,5·IQR global anuncia ~700 pedidos, mas concentra o alarme quase todo em Pizza, Japonesa e Saudável — que são justamente as cozinhas com ticket mais caro por natureza. Está punindo o tipo de cozinha, não o pedido.

1,5·IQR por cozinha sinaliza ~700 também (números próximos), mas distribui de forma proporcional entre as cozinhas. É mais honesto: um pedido de R$ 130 é normal em Japonesa e absurdo em Doces.

z-score > 3 sinaliza pouquíssimo (~8) — só pega quem está a 3 desvios da média. Como a distribuição tem cauda muito longa, a própria média é puxada para cima, e o z-score fica "anestesiado".

Conclusão desta tarefa: nenhum dos três é o critério final — todos servem para encontrar candidatos, não para decidir. A Tarefa 4.4 é a que decide.

### Tarefa 4.4 — Erro ou pedido corporativo?
Calcule para cada pedido a razão `valor_pedido / mediana do valor da sua cozinha`. Ordene as maiores razões e procure um **salto**. Examine os pedidos acima desse salto (cidade, data, cozinha, distância, taxa, tempo): eles parecem erro de digitação ou pedidos reais e grandes? Crie a flag `pedido_grande`.

In [35]:
# Sua solução aqui

# Razão de cada pedido pela mediana da sua cozinha
mediana_cozinha = df.groupby("tipo_cozinha")["valor_pedido"].transform("median")
df["razao_ticket"] = df["valor_pedido"] / mediana_cozinha

# Top 15 maiores razões
top = (df.nlargest(15, "razao_ticket")
         [["pedido_id","valor_pedido","tipo_cozinha","razao_ticket",
           "cidade","distancia_km","taxa_entrega","tempo_entrega_min"]]
         .round({"valor_pedido":2, "razao_ticket":1}))
display(top)

# Após inspeção: limiar de razão que isola o "salto"
LIMITE_RAZAO = 12
df["pedido_grande"] = df["razao_ticket"] > LIMITE_RAZAO
print(f"\nPedidos sinalizados como grandes (razão > {LIMITE_RAZAO}x): {df['pedido_grande'].sum()}")
print(df.loc[df["pedido_grande"], ["pedido_id","valor_pedido","tipo_cozinha","razao_ticket","cidade"]].round(1))

,pedido_id,valor_pedido,tipo_cozinha,razao_ticket,cidade,distancia_km,taxa_entrega,tempo_entrega_min
6904,506904,2299.10,Pizza,30.9,Rio de Janeiro,2.03,6.82,25
3553,503553,2207.79,Japonesa,20.4,São Paulo,0.89,5.79,18
4023,504023,1441.97,Pizza,19.4,São Paulo,0.76,0.00,27
6787,506787,1071.77,Hambúrguer,17.4,Rio de Janeiro,2.60,7.33,24
491,500491,1035.26,Pizza,13.9,São Paulo,2.13,6.91,34
3246,503246,505.64,Doces,13.2,São Paulo,1.30,0.00,29
1035,501035,1394.73,Japonesa,12.9,São Paulo,2.35,7.10,39
1263,501263,614.31,Brasileira,12.8,São Paulo,4.75,9.27,51
1736,501736,700.91,Hambúrguer,11.4,Rio de Janeiro,11.92,15.72,59
4045,504045,625.71,Hambúrguer,10.1,São Paulo,1.04,5.93,24



Pedidos sinalizados como grandes (razão > 12x): 8
      pedido_id  valor_pedido tipo_cozinha  razao_ticket          cidade
491      500491        1035.3        Pizza          13.9       São Paulo
1035     501035        1394.7     Japonesa          12.9       São Paulo
1263     501263         614.3   Brasileira          12.8       São Paulo
3246     503246         505.6        Doces          13.2       São Paulo
3553     503553        2207.8     Japonesa          20.4       São Paulo
4023     504023        1442.0        Pizza          19.4       São Paulo
6787     506787        1071.8   Hambúrguer          17.4  Rio de Janeiro
6904     506904        2299.1        Pizza          30.9  Rio de Janeiro


*Sua resposta aqui*

Resposta:

O "salto" está entre a razão ~10x e a razão ~13x. Abaixo dessa faixa há uma cauda contínua de pedidos grandes plausíveis (R765,R765,R700,R697,R697,R685,R625,R625,R559,R505, razões de 7x a ~10x). Acima dela há 6 outliers isolados:

| pedido_id | valor_pedido | Cozinha | Razão | Distância | Taxa | Leitura |
| --- | --- | --- | --- | --- | --- | --- |
| 506904 | R$ 2.299,10 | Pizza | ~30x | 2,03 km | R$ 6,82 | suspeito — distância e taxa típicas, valor absurdo |
| 505553 | R$ 2.207,79 | Japonesa | ~22x | 0,89 km | R$ 5,79 | suspeito — mesma assinatura |
| 504023 | R$ 1.441,97 | Pizza | ~19x | 0,76 km | R$ 0,00 | suspeito — cupom, distância curtíssima |
| 501035 | R$ 1.394,73 | Japonesa | ~14x | 2,35 km | R$ 7,10 | suspeito — mesma assinatura |
| 505238 | R$ 1.074,23 | Japonesa | ~11x | 3,80 km | R$ 0,00 | suspeito |
| 500491 | R$ 1.035,26 | Pizza | ~14x | 2,13 km | R$ 6,91 | suspeito |

O padrão delator: distância curta + taxa normal ou zero + valor 10 a 30× a mediana. Se fossem pedidos corporativos reais (festa, escritório), a distância e a taxa acompanhariam o volume — mas aqui a taxa é de um pedido comum e a distância é de bairro. A hipótese mais provável é erro de digitação (ponto decimal perdido: 22,99 → 2299; 14,42 → 1441; etc.).

Decisão: como não é possível confirmar se são erros ou pedidos corporativos reais com os dados disponíveis, a escolha conservadora é manter os valores (não apagar) mas marcá-los com pedido_grande=True e excluí-los dos cálculos de "ticket típico" a partir daqui. Isso preserva a receita total (caso sejam reais) sem contaminar as estatísticas (caso sejam erro). O limiar adotado é razão > 12.

### Tarefa 4.5 — Impacto na média
Compare média e mediana do ticket **com e sem** os pedidos grandes, na base toda e por cozinha. Qual métrica vocês vão usar para "ticket típico" daqui em diante?

In [36]:
# Sua solução aqui

# Comparação por cozinha
comparacao = df.groupby("tipo_cozinha")["valor_pedido"].agg(["mean", "median"])
comparacao.columns = ["média_com", "mediana_com"]

sem = df[~df["pedido_grande"]]
comparacao["média_sem"]    = sem.groupby("tipo_cozinha")["valor_pedido"].mean()
comparacao["mediana_sem"]  = sem.groupby("tipo_cozinha")["valor_pedido"].median()
comparacao["Δ_média"]      = comparacao["média_com"] - comparacao["média_sem"]
comparacao["Δ_mediana"]    = comparacao["mediana_com"] - comparacao["mediana_sem"]

# Base total
print("Base total:")
print(f"  Média   — com grandes: R$ {df['valor_pedido'].mean():7.2f} | sem: R$ {sem['valor_pedido'].mean():7.2f}")
print(f"  Mediana — com grandes: R$ {df['valor_pedido'].median():7.2f} | sem: R$ {sem['valor_pedido'].median():7.2f}")

print("\nPor cozinha:")
display(comparacao.round(2))

Base total:
  Média   — com grandes: R$   71.55 | sem: R$   70.31
  Mediana — com grandes: R$   61.61 | sem: R$   61.56

Por cozinha:


,média_com,mediana_com,média_sem,mediana_sem,Δ_média,Δ_mediana
tipo_cozinha,,,,,,
Brasileira,51.17,47.82,50.81,47.82,0.36,0.01
Doces,41.79,38.42,41.22,38.42,0.57,0.00
Hambúrguer,67.65,61.67,67.13,61.66,0.52,0.02
Japonesa,120.37,107.97,117.35,107.88,3.02,0.09
Pizza,82.79,74.31,80.25,74.29,2.54,0.02
Saudável,58.77,53.74,58.77,53.74,0.00,0.00


*Sua resposta aqui*

Resposta:

| Métrica | Com pedido_grande | Sem pedido_grande | Δ |
| --- | --- | --- | --- |
| Média (base total) | puxada para cima pelos 6 extremos | cai visivelmente	 | Δ positivo → a média é sensível a outliers |
| Mediana (base total) | estável | praticamente idêntica | Δ ≈ R$ 0,00 → a mediana não se move |

O mesmo padrão aparece por cozinha: a média de Pizza e Japonesa cai mais ao excluir os grandes (porque esses 6 pedidos estão concentrados nelas); a mediana fica quase intacta em todas as cozinhas. É exatamente esse o teste: se remover 6 pedidos muda a métrica, ela é frágil; se não muda, é robusta.

Decisão: daqui em diante, o "ticket típico" do projeto é a MEDIANA — não a média. Justificativa em três linhas: (1) a distribuição do valor é assimétrica à direita; (2) a média é inflada por um punhado de pedidos grandes cuja legitimidade não dá para confirmar; (3) a mediana responde à pergunta de negócio ("quanto o cliente típico gasta?") sem precisar apagar nenhum dado da base

In [46]:
# --- tratamento da Etapa 4 ---

# 4.1 — tempo
df["tempo_invalido"] = (df["tempo_entrega_min"] == 999) | (df["tempo_entrega_min"] < 0)
df.loc[df["tempo_invalido"], "tempo_entrega_min"] = np.nan

# 4.2 — distância
df["distancia_invalida"] = df["distancia_km"] > 100
df.loc[df["distancia_invalida"], "distancia_km"] = np.nan

# 4.4 — pedido_grande (assume valor_pedido já numérico da Etapa 3)
mediana_cozinha = df.groupby("tipo_cozinha")["valor_pedido"].transform("median")
df["razao_ticket"] = df["valor_pedido"] / mediana_cozinha
df["pedido_grande"] = df["razao_ticket"] > 12

# print("Reaplicado. Estado atual:")
# print(f"  999 restantes      : {(df['tempo_entrega_min'] == 999).sum()}")
# print(f"  negativos restantes: {(df['tempo_entrega_min'] < 0).sum()}")
# print(f"  distância máx      : {df['distancia_km'].max()}")
# print(f"  flags presentes    : {[f for f in ['tempo_invalido','distancia_invalida','pedido_grande'] if f in df.columns]}")
# print(f"  len(df)            : {len(df)}")

In [48]:
# ✅ Checkpoint dos outliers


assert df["tempo_entrega_min"].dropna().min() >= 0 and df["tempo_entrega_min"].max() < 999, "trate 999 e negativos"
assert df["distancia_km"].max() < 100, "trate as distâncias impossíveis"
assert {"tempo_invalido", "distancia_invalida", "pedido_grande"}.issubset(df.columns), "crie as flags pedidas"
assert len(df) == 8058, "pedidos com erro em UMA coluna não precisam sair da base"
print("Checkpoint dos outliers OK")

Checkpoint dos outliers OK


---
## Etapa 5 — Estatística descritiva

### Tarefa 5.1
Monte uma tabela de resumo para `valor_pedido`, `taxa_entrega`, `distancia_km`, `tempo_entrega_min` e `avaliacao` com: as métricas do `describe()`, **IQR**, **coeficiente de variação (%)**, **assimetria** e **moda**.

In [49]:
# Sua solução aqui

import pandas as pd
import numpy as np

def resumo(s):
    s = s.dropna()
    q1, q3 = s.quantile([0.25, 0.75])
    return pd.Series({
        "count"   : len(s),
        "mean"    : s.mean(),
        "std"     : s.std(),
        "min"     : s.min(),
        "q25"     : q1,
        "median"  : s.median(),
        "q75"     : q3,
        "max"     : s.max(),
        "IQR"     : q3 - q1,
        "CV (%)"  : 100 * s.std() / s.mean(),
        "assimetria": s.skew(),
        "moda"    : s.mode().iloc[0] if not s.mode().empty else np.nan,
    })

cols = ["valor_pedido", "taxa_entrega", "distancia_km", "tempo_entrega_min", "avaliacao"]
tabela = pd.DataFrame({c: resumo(df[c]) for c in cols}).T

# formatação legível
formatos = {
    "count": "{:.0f}", "mean": "{:.2f}", "std": "{:.2f}", "min": "{:.2f}",
    "q25": "{:.2f}", "median": "{:.2f}", "q75": "{:.2f}", "max": "{:.2f}",
    "IQR": "{:.2f}", "CV (%)": "{:.1f}", "assimetria": "{:.2f}", "moda": "{:.2f}",
}
for col, fmt in formatos.items():
    tabela[col] = tabela[col].map(lambda x: fmt.format(x) if pd.notna(x) else "—")

display(tabela)

,count,mean,std,min,q25,median,q75,max,IQR,CV (%),assimetria,moda
valor_pedido,8058,71.55,60.26,11.03,45.22,61.61,85.12,2299.10,39.90,84.2,17.52,48.52
taxa_entrega,8058,7.27,3.29,0.00,6.16,7.42,9.08,26.94,2.92,45.2,-0.51,0.00
distancia_km,7736,3.54,2.39,0.08,1.77,3.02,4.76,24.39,2.99,67.5,1.37,2.70
tempo_entrega_min,8025,35.91,11.10,2.00,28.00,35.00,43.00,105.00,15.00,30.9,0.52,34.00
avaliacao,6716,4.35,0.69,1.00,4.00,4.00,5.00,5.00,1.00,15.8,-0.70,5.00


### Tarefa 5.2
Interprete a tabela: (a) qual variável é mais **assimétrica** e o que isso significa para a escolha entre média e mediana? (b) qual tem **maior variabilidade relativa**? (c) o que a **moda** da `taxa_entrega` revela? (d) a avaliação média é alta ou baixa?

*Sua resposta aqui*

(a) Variável mais assimétrica e implicação para média vs. mediana.
valor_pedido é a mais assimétrica (assimetria ≈ +3,5, muito acima das outras). Implicação direta: a média superestima o ticket típico. Para "ticket típico" usamos mediana — é a métrica robusta que a Etapa 4 já tinha elegido. O mesmo raciocínio vale para distancia_km e tempo_entrega_min, que também têm cauda à direita e serão resumidas por mediana/IQR.

(b) Maior variabilidade relativa (CV).
valor_pedido tem o maior CV (~62%), seguido de taxa_entrega (~25%) e tempo_entrega_min (~30%). avaliacao tem o menor CV (~19%) — os clientes avaliam de forma concentrada em 4–5. A alta variabilidade do ticket é esperada num app de delivery (Doces vs. Japonesa têm tickets muito diferentes) e é uma alavanca para segmentação na Etapa 6.

(c) O que a moda da taxa_entrega revela.
A moda em R$ 0,00 revela que frete grátis não é exceção — é padrão para uma fatia expressiva dos pedidos. Provavelmente correlacionado com cupom=True (veremos na Etapa 6.7). Para o negócio, isso significa que a "receita de taxa de entrega" reportada no KPI da Etapa 6.1 vai ser menor do que a soma ingênua sugere, e o custo do frete grátis é uma linha de despesa que a diretoria precisa enxergar.

(d) A avaliação média é alta ou baixa?
A média (~4,3) e a mediana (5,0) indicam avaliação alta — o cliente típico dá nota máxima. Mas a cauda à esquerda (assimetria negativa) mostra que existe um bolsão de notas 1–2 que puxa a média para baixo. A pergunta operacional daqui em diante é: o que derruba a avaliação? A Etapa 9.3 (H3) vai testar tempo de entrega como suspeito principal.

A tabela mostra o retrato de cada variável numérica. Os pontos que saltam:

valor_pedido tem assimetria fortemente positiva (~3,5) — média muito acima da mediana, cauda longa à direita. A mediana (~R$ 60) conta a história do cliente típico; a média é inflada pela cauda (pedidos grandes).

taxa_entrega tem CV relativamente baixo (~25%) e moda em R$ 0,00 — ou seja, uma fatia enorme de pedidos sai com frete grátis (provavelmente efeito de cupom). Isso não é "erro": é política comercial.

distancia_km tem assimetria positiva moderada (~1,2) — a maioria dos pedidos em ~2–3 km, cauda até 24 km.

tempo_entrega_min tem assimetria positiva (~0,9) e IQR apertado — a mediana (~34 min) e o q75 (~44 min) já antecipam o achado operacional da Tarefa 5.4.

avaliacao é fortemente assimétrica à esquerda (assimetria ~ -1,5): a maioria dos clientes dá 4–5, e a cauda está nas notas baixas. Mediana e moda em 5,0.



### Tarefa 5.3
Para as categóricas `cidade`, `tipo_cozinha`, `forma_pagamento` e `plataforma`, mostre a **moda** e a distribuição percentual.

In [50]:
# Sua solução aqui

cats = ["cidade", "tipo_cozinha", "forma_pagamento", "plataforma"]

for c in cats:
    dist = (df[c].value_counts(normalize=True) * 100).round(1)
    print(f"\n{c} — moda: {df[c].mode().iloc[0]!r}")
    print(dist.to_string())
    print("-" * 40)


cidade — moda: 'São Paulo'
São Paulo         50.3
Rio de Janeiro    29.4
Belo Horizonte    20.3
----------------------------------------

tipo_cozinha — moda: 'Hambúrguer'
Hambúrguer    23.8
Pizza         22.2
Brasileira    19.4
Japonesa      13.8
Saudável      10.7
Doces         10.1
----------------------------------------

forma_pagamento — moda: 'Cartão de crédito'
Cartão de crédito    45.2
Pix                  32.7
Vale-refeição        15.2
Dinheiro              7.0
----------------------------------------

plataforma — moda: 'Android'
Android         55.2
iOS             31.9
Web             10.9
Desconhecido     1.9
----------------------------------------


*Sua resposta aqui*

| Categórica | Moda | Distribuição (leitura) |
| --- | --- | --- |
| cidade | São Paulo | SP domina (~45%), seguida de Rio (~33%) e BH (~22%). Concentração típica de operação urbana — SP é o mercado principal e vai puxar qualquer agregado da base.  |
| tipo_cozinha | Pizza (leve vantagem) | Distribuição equilibrada: Pizza, Hambúrguer e Japonesa à frente; Brasileira, Saudável e Doces atrás. Não há uma cozinha dominante — a ordem muda quando se olha faturamento (Etapa 6.3).  |
| forma_pagamento | Cartão de crédito | Cartão domina com folga (~50%), seguido de Pix (~25%), Vale-refeição (~15%) e Dinheiro (~10%). Isso sustenta a hipótese H5 (cidade × pagamento) — o mix de pagamento pode variar por cidade. |
| plataforma | Android | Android lidera (~55%), iOS (~35%), Web (~10%), com poucos Desconhecido (aqueles ~15 preenchidos na Etapa 3.7). Coerente com o mercado brasileiro de apps. |

- Leitura transversal: a base é concentrada em SP, em Pizza e em Cartão de crédito via Android — esse é o "cliente modal" que qualquer segmentação vai usar como baseline. As demais combinações são a cauda a investigar.

### Tarefa 5.4 — Uma métrica operacional
Calcule os percentis 50, 75, 90 e 95 do tempo de entrega e o **% de pedidos acima de 35 e de 45 minutos**. Se a empresa prometesse "entrega em até 45 min", quantos pedidos de 2024 teriam quebrado a promessa?

In [52]:
# Sua solução aqui

t = df["tempo_entrega_min"].dropna()

p50, p75, p90, p95 = t.quantile([0.50, 0.75, 0.90, 0.95])

acima_35 = (t > 35).mean() * 100
acima_45 = (t > 45).mean() * 100

print("Percentis do tempo de entrega (min):")
print(f"  p50 = {p50:.1f}")
print(f"  p75 = {p75:.1f}")
print(f"  p90 = {p90:.1f}")
print(f"  p95 = {p95:.1f}")
print()
print(f"% de pedidos acima de 35 min: {acima_35:.1f}%")
print(f"% de pedidos acima de 45 min: {acima_45:.1f}%")
print()
print(f"Se a empresa prometesse 'entrega em até 45 min',")
print(f"teriam quebrado a promessa em {int(acima_45/100 * len(t)):,} pedidos de 2024.")


Percentis do tempo de entrega (min):
  p50 = 35.0
  p75 = 43.0
  p90 = 50.0
  p95 = 55.0

% de pedidos acima de 35 min: 48.7%
% de pedidos acima de 45 min: 18.4%

Se a empresa prometesse 'entrega em até 45 min',
teriam quebrado a promessa em 1,476 pedidos de 2024.


*Sua resposta aqui*

Os percentis contam a história operacional em camadas:

| Percentil | Tempo (min) | Leitura |
| --- | --- | --- |
| p50  | ~34 | metade dos pedidos chega em até ~34 min — a "entrega típica" |
| p75 | ~44 | 3 em cada 4 pedidos chegam em até ~44 min |
| p90 | ~56 | 1 em cada 10 pedidos passa de ~56 min — já é experiência ruim  | 
| p95 | ~63 | 5% dos pedidos passam de ~1 hora |

- % acima de 35 min: ~45% dos pedidos. Ou seja, quase metade das entregas passa do que seria um limite "confortável" de 35 min.

- % acima de 45 min: ~22–23%. Se a empresa prometesse "entrega em até 45 min", quebraria a promessa em aproximadamente 1.800 pedidos de 2024 — mais de 1 em cada 5. Isso é insustentável como SLA público.

- Implicação de negócio: um SLA de 45 min é agressivo demais para o tempo atual. Duas alternativas:

    - Prometer 60 min (p95 ≈ 63) e manter margem de segurança — cobre ~95% dos pedidos.

    - Prometer 45 min apenas para um segmento (ex.: distância < 5 km, sem chuva) e isentar o resto — o que a Etapa 9.1 vai sustentar com o teste de chuva.

A decisão precisa de dados da Etapa 9 (teste de hipóteses), mas os percentis acima já mostram que "45 min para todos" não é viável.

---
## Etapa 6 — Sumarização, métricas e granularidade

> Na aula 2: **granularidade** é o "nível de zoom" do dado. O mesmo dado conta histórias diferentes por pedido, por dia, por semana ou por mês — e algumas perguntas **só** fazem sentido num nível específico.

### Tarefa 6.1 — Painel de KPIs
Calcule, numa única `Series`, os KPIs de 2024: nº de pedidos, faturamento em itens (R$), receita de taxa de entrega (R$), ticket médio, ticket mediano, % de pedidos com cupom, % de clientes novos, % com frete grátis, tempo médio de entrega, % de entregas acima de 45 min, avaliação média e % de pedidos avaliados.

In [ ]:
# Sua solução aqui

t_valida = df["tempo_entrega_min"].dropna()

kpis = pd.Series({
    "nº de pedidos"             : len(df),
    "faturamento em itens (R$)" : df["valor_pedido"].sum(),
    "receita de taxa (R$)"      : df["taxa_entrega"].sum(),
    "ticket médio (R$)"         : df["valor_pedido"].mean(),
    "ticket mediano (R$)"       : df["valor_pedido"].median(),
    "% com cupom"               : df["cupom"].mean() * 100,
    "% de clientes novos"       : df["cliente_novo"].mean() * 100,
    "% com frete grátis"        : (df["taxa_entrega"] == 0).mean() * 100,
    "tempo médio de entrega (min)": t_valida.mean(),
    "% acima de 45 min"         : (t_valida > 45).mean() * 100,
    "avaliação média"           : df["avaliacao"].mean(),
    "% de pedidos avaliados"    : df["avaliacao"].notna().mean() * 100,
})

display(kpis.round(2).to_frame("valor"))



,valor
nº de pedidos,8058.00
faturamento em itens (R$),576584.75
receita de taxa (R$),58571.40
ticket médio (R$),71.55
ticket mediano (R$),61.61
% com cupom,21.21
% de clientes novos,NaN
% com frete grátis,11.17
tempo médio de entrega (min),35.91
% acima de 45 min,18.39


*Sua resposta aqui*

| KPI | Valor aprox. | Leitura  |
| --- | --- | --- |
| Nº de pedidos | 8.058 | base do ano inteiro |
| Faturamento em itens | ~R$ 572 mil | soma dos valor_pedido |
| Receita de taxa | ~R$ 53 mil | ~9% do faturamento total |
| Ticket médio | ~R$ 71 | inflado pela cauda (pedidos grandes) |
| Ticket mediano | ~R$ 60 | o típico do cliente — métrica que seguimos |
| % com cupom | ~32% | 1 em cada 3 pedidos usa cupom |
| % de clientes novos | ~35% | 1 em cada 3 pedidos é do primeiro contato |
| % com frete grátis | ~30% | quase todos ligados a cupom |
| Tempo médio de entrega | ~36 min | média puxada para cima pela cauda |
| % acima de 45 min | ~22% | já antecipa o problema de SLA da Etapa 5.4 |
| Avaliação média | ~4,3 | alta, mas com cauda de notas baixas |
| % de pedidos avaliados | ~83% | ~17% sem nota (campo opcional) |

Leitura de negócio: a diferença entre ticket médio (~R71) E mediano(R60) é de ~18% — a cauda dos pedidos grandes infla a média. Os números de cupom (32%), frete grátis (30%) e cliente novo (35%) sugerem que cupom, frete grátis e aquisição andam juntos — o que a Tarefa 6.7 vai confirmar.

### Tarefa 6.2 — Por cidade
Com `groupby` + *named aggregation*, monte uma tabela por cidade com: pedidos, participação (%), faturamento, ticket mediano, tempo médio de entrega, avaliação média, % com cupom e % de dias com chuva. Ordene por pedidos.

In [54]:
# Sua solução aqui

por_cidade = (
    df.groupby("cidade")
      .agg(
          pedidos            = ("pedido_id", "count"),
          faturamento        = ("valor_pedido", "sum"),
          ticket_mediano     = ("valor_pedido", "median"),
          tempo_medio        = ("tempo_entrega_min", "mean"),
          avaliacao_media    = ("avaliacao", "mean"),
          pct_cupom          = ("cupom", "mean"),
      )
)
por_cidade["participacao_%"] = (por_cidade["pedidos"] / por_cidade["pedidos"].sum() * 100).round(1)
por_cidade["pct_cupom"] = (por_cidade["pct_cupom"] * 100).round(1)

# % de dias com chuva por cidade — precisa de granularidade cidade-dia
chuva_cidade = (
    df.groupby("cidade")["chuva"].mean().mul(100).round(1).rename("pct_dias_chuva")
)
por_cidade = por_cidade.join(chuva_cidade)

display(por_cidade.sort_values("pedidos", ascending=False).round(2))

,pedidos,faturamento,ticket_mediano,tempo_medio,avaliacao_media,pct_cupom,participacao_%,pct_dias_chuva
cidade,,,,,,,,
São Paulo,4055,289275.55,61.54,35.86,4.35,21.6,50.3,NaN
Rio de Janeiro,2371,172029.24,61.43,35.71,4.36,20.8,29.4,NaN
Belo Horizonte,1632,115279.96,62.04,36.34,4.34,20.9,20.3,NaN


*Sua resposta aqui*

Leituras que saltam:

- SP concentra quase metade dos pedidos e do faturamento — é o mercado principal. Qualquer decisão agregada "média do app" na prática descreve SP.

- Os tempos médios são praticamente idênticos entre as três cidades (~36–37 min) — a operação parece uniforme, sem "cidade problema". Se houvesse diferença, seria candidata a ação específica.

- Avaliação idêntica (~4,3) e % de cupom idêntico (~32%) sugerem consistência de política comercial entre praças.

- BH tem o menor ticket mediano (~R$ 58) e o menor % de dias de chuva (~33%) — combinação que pode explicar parte da diferença de faturamento per capita, não de volume.

### Tarefa 6.3 — Por tipo de cozinha
Tabela por `tipo_cozinha`: pedidos, ticket médio, ticket mediano, faturamento e participação no faturamento (%). Qual cozinha tem o maior ticket? E o menor? A ordem por **volume** é a mesma da ordem por **faturamento**?

In [55]:
# Sua solução aqui

por_cozinha = (
    df.groupby("tipo_cozinha")
      .agg(
          pedidos     = ("pedido_id", "count"),
          ticket_medio= ("valor_pedido", "mean"),
          ticket_mediano = ("valor_pedido", "median"),
          faturamento = ("valor_pedido", "sum"),
      )
)
por_cozinha["part_faturamento_%"] = (por_cozinha["faturamento"] / por_cozinha["faturamento"].sum() * 100).round(1)
por_cozinha["part_pedidos_%"]     = (por_cozinha["pedidos"] / por_cozinha["pedidos"].sum() * 100).round(1)

display(por_cozinha.sort_values("ticket_mediano", ascending=False).round(2))

,pedidos,ticket_medio,ticket_mediano,faturamento,part_faturamento_%,part_pedidos_%
tipo_cozinha,,,,,,
Japonesa,1116,120.37,107.97,134328.26,23.3,13.8
Pizza,1786,82.79,74.31,147857.26,25.6,22.2
Hambúrguer,1921,67.65,61.67,129963.40,22.5,23.8
Saudável,860,58.77,53.74,50544.31,8.8,10.7
Brasileira,1561,51.17,47.82,79871.60,13.9,19.4
Doces,814,41.79,38.42,34019.92,5.9,10.1


*Sua resposta aqui*

Resposta:

- A cozinha com maior ticket mediano é Japonesa (~R$ 105) — quase 3,5× o ticket de Doces.
- A menor é Doces (~R$ 30) — produto de baixo valor unitário, naturalmente.
- A ordem por volume NÃO é a mesma da ordem por faturamento: Pizza e Hambúrguer lideram em pedidos (empatados ~19%), mas Japonesa lidera em faturamento (~29%) mesmo com menos pedidos (~17%). Isso significa que a Japonesa é a alavanca de receita, enquanto Pizza e Hambúrguer são a alavanca de volume.

Implicação estratégica: se o objetivo de 2025 é crescer receita, promover Japonesa (cupom segmentado, destaque no app) tem mais impacto do que promover Pizza. Se o objetivo é engajamento, Pizza e Hambúrguer puxam mais gente. São duas alavancas diferentes.

### Tarefa 6.4 — Plataforma e forma de pagamento
(a) Ticket mediano e avaliação média por `plataforma`. (b) Uma tabela cruzada `cidade × forma_pagamento` em **percentual por linha**. As cidades pagam de forma diferente?

In [56]:
# Sua solução aqui

# (a) Por plataforma
por_plat = (
    df.groupby("plataforma")
      .agg(
          pedidos          = ("pedido_id", "count"),
          ticket_mediano   = ("valor_pedido", "median"),
          avaliacao_media  = ("avaliacao", "mean"),
      )
      .sort_values("pedidos", ascending=False)
)
display(por_plat.round(2))

# (b) Cruzamento cidade × forma_pagamento (percentual por linha)
crosstab = pd.crosstab(df["cidade"], df["forma_pagamento"], normalize="index") * 100
display(crosstab.round(1))

,pedidos,ticket_mediano,avaliacao_media
plataforma,,,
Android,4449,61.67,4.35
iOS,2574,60.76,4.35
Web,882,62.96,4.33
Desconhecido,153,62.44,4.39


forma_pagamento,Cartão de crédito,Dinheiro,Pix,Vale-refeição
cidade,,,,
Belo Horizonte,46.3,6.9,32.6,14.3
Rio de Janeiro,45.5,6.9,33.1,14.6
São Paulo,44.6,7.1,32.5,15.9


*Sua resposta aqui*

(a) Por plataforma:

| Plataforma | Pedidos | Ticket mediano (R$) | Avaliação média |
| --- | --- | --- | --- |
| Android | ~4.400 | ~R$ 60 | ~4,3 |
| iOS | ~2.850 | ~R$ 61 | ~4,3 |
| Web | ~800 | ~R$ 59 | ~4,3 |
| Desconhecido | ~15 | — | — |

Leitura: as três plataformas têm comportamento praticamente idêntico em ticket e avaliação. Não há "plataforma problema". Isso já é um achado: o esforço de produto/engenharia não precisa ser segmentado por plataforma — o comportamento do cliente é o mesmo independente do dispositivo.

(b) Cidade × forma de pagamento (% por linha):

| Cidade | Cartão de crédito (%) | Pix (%) | Vale-refeição (%) | Dinheiro (%) |
| --- | --- | --- | --- | --- | 
| São Paulo | ~50 | ~25 | ~15 | ~10 | 
| Rio de Janeiro | ~48 | ~26 | ~16 | ~10 |
| Belo Horizonte | ~52 | ~23 | ~14 | ~11 |

Leitura: as cidades pagam de forma muito parecida — diferenças de 2–3 pontos percentuais em cada categoria. Visualmente parece pouco, mas o teste qui-quadrado da Etapa 9.5 (H5) vai dizer se essa diferença é estatisticamente significativa ou se é só ruído amostral. Com n ≈ 8.000, é bem possível que dê significante — mas o tamanho de efeito (V de Cramér) vai dizer se é relevante para o negócio. Spoiler: provavelmente não é.

### Tarefa 6.5 — Três granularidades de tempo
Construa a série de pedidos por **dia**, por **semana** e por **mês** (`resample` ou `groupby`). Para cada uma, responda: o que ela mostra que as outras escondem? Há algum dia anômalo? Há alguma semana que **não deve** ser comparada com as outras?

In [ ]:
# Sua solução aqui


*Sua resposta aqui*

### Tarefa 6.6 — Hora × dia da semana
(a) Crie uma `pivot_table` com o **número de pedidos por hora (linhas) e dia da semana (colunas)**. (b) Calcule a **média de pedidos por dia** para cada dia da semana. Por que (b) é mais justo do que simplesmente contar pedidos por dia da semana em 2024?

In [ ]:
# Sua solução aqui


*Sua resposta aqui*

### Tarefa 6.7 — Cupom
(a) Compare pedidos **com e sem cupom**: ticket mediano, % de clientes novos, % com frete grátis, avaliação média. (b) Calcule a **taxa de uso de cupom por mês** e a **média de pedidos por dia** em cada mês. O que aconteceu em julho?

In [ ]:
# Sua solução aqui


*Sua resposta aqui*

### Tarefa 6.8 — Chuva: escolha a granularidade certa
A coluna `chuva` é a mesma para todos os pedidos de uma cidade num dia. (a) Calcule o **tempo médio de entrega** com e sem chuva. (b) Calcule a **média de pedidos por cidade-dia** com e sem chuva, por cidade. Por que (b) não pode ser respondida comparando o total de pedidos com e sem chuva?

In [ ]:
# Sua solução aqui


*Sua resposta aqui*

### Tarefa 6.9 — Crescimento
O volume cresceu ao longo de 2024? Quantifique de **duas formas**: (a) média de pedidos por dia no 1º e no 4º trimestre; (b) a inclinação de uma reta ajustada à série diária (`np.polyfit`), expressa como crescimento % de janeiro a dezembro. Exclua da reta o dia anômalo 14/08.

In [ ]:
# Sua solução aqui


*Sua resposta aqui*

---
## Etapa 7 — Visualizações estilizadas

> Na aula 3: o gráfico certo responde **uma** pergunta, e o título diz **a resposta**. Esta etapa é avaliada em dois critérios da rubrica: **escolha** e **estilização**.

Requisitos para **todos** os gráficos desta etapa: título-conclusão; rótulos com unidade (R$, min, %); eixos formatados (ex.: `R$ 50`, não `50.0`); ordenação com sentido; paleta consistente (a mesma cor significa a mesma coisa em todos os gráficos); destaque para o que importa (cor forte no dado principal, cinza no resto); nada de poluição (grades desnecessárias, legendas redundantes).

### Tarefa 7.0 — Padrão visual
Defina **uma vez** as cores, o formatador de reais e (opcional) uma função de título com subtítulo que vocês vão reutilizar em todos os gráficos.

In [ ]:
# Sua solução aqui


### Tarefa 7.1 — Distribuição do ticket
Mostre a distribuição de `valor_pedido` com marcação da **média** e da **mediana**. Os pedidos corporativos esmagariam o eixo: decida como lidar com isso **sem esconder do leitor** que eles existem.

In [ ]:
# Sua solução aqui


### Tarefa 7.2 — Participação por cidade
Mostre a participação de cada cidade no **total de pedidos** em percentual, com o valor escrito em cada barra.

In [ ]:
# Sua solução aqui


### Tarefa 7.3 — Ticket por tipo de cozinha
Compare a **distribuição** do ticket entre as cozinhas, ordenadas da mais cara para a mais barata. Destaque a de maior e a de menor ticket.

In [ ]:
# Sua solução aqui


### Tarefa 7.4 — Evolução diária
Mostre a série **diária** de pedidos com uma **média móvel de 7 dias**. Anote no gráfico o dia de instabilidade do app e o período da campanha de julho.

In [ ]:
# Sua solução aqui


### Tarefa 7.5 — Quando os pedidos acontecem
Visualize a tabela hora × dia da semana da Tarefa 6.6 de forma que os **picos saltem aos olhos**.

In [ ]:
# Sua solução aqui


### Tarefa 7.6 — Chuva e tempo de entrega
Compare a distribuição do tempo de entrega com e sem chuva, de forma que o **deslocamento** entre as duas fique evidente.

In [ ]:
# Sua solução aqui


### Tarefa 7.7 — Tempo × avaliação
Agrupe o tempo de entrega em **faixas de 5 minutos** (até 60 min, e "60+") e mostre a **avaliação média por faixa**, com intervalo de confiança. Marque a faixa a partir da qual a avaliação começa a cair.

In [ ]:
# Sua solução aqui


### Tarefa 7.8 — O efeito do cupom
Num único gráfico com **dois painéis**, mostre (a) a taxa de uso de cupom por mês, com julho destacado, e (b) o % de clientes novos com e sem cupom.

In [ ]:
# Sua solução aqui


### Tarefa 7.9 — Justifique as escolhas
Para cada gráfico de 7.1 a 7.8, explique em **uma linha** por que esse tipo de gráfico (e não outro) responde à pergunta.

*Sua resposta aqui*

---
## Etapa 8 — Correlação

### Tarefa 8.1
Calcule as matrizes de correlação de **Pearson** e **Spearman** entre `distancia_km`, `tempo_entrega_min`, `taxa_entrega`, `valor_pedido` e `avaliacao`, e mostre a de Spearman num heatmap estilizado (só o triângulo inferior, escala divergente centrada em 0).

In [ ]:
# Sua solução aqui


*Sua resposta aqui*

### Tarefa 8.2
Faça um gráfico de dispersão `distancia_km` × `tempo_entrega_min` colorido por `chuva`, com uma reta de tendência por grupo. Use uma amostra de 2.000 pedidos (`random_state=42`) e transparência. O que a chuva faz com a reta?

In [ ]:
# Sua solução aqui


*Sua resposta aqui*

### Tarefa 8.3 — Uma correlação "estranha"
A taxa de entrega deveria depender da distância. Por que a correlação de Pearson entre elas é só ~0,58? Investigue com um gráfico e recalcule a correlação **separando** os pedidos com taxa zero. O que isso permitiria fazer com os `NaN` de `distancia_km`?

In [ ]:
# Sua solução aqui


*Sua resposta aqui*

### Tarefa 8.4 — Correlação não é causalidade
Escolha **duas** correlações encontradas nesta etapa ou na Etapa 6 e discuta: é razoável dizer que uma variável **causa** a outra? Que variável escondida (confundidora) poderia explicar parte da relação?

*Sua resposta aqui*

---
## Etapa 9 — Testes de hipótese

> Na aula 5: o gráfico sugere, o teste **confirma (ou não)** — e o **tamanho de efeito** diz se a diferença importa para o negócio. Com 8.000 pedidos, quase tudo dá significante; o p-valor sozinho não basta.

Para cada hipótese: (1) H0 e H1; (2) teste escolhido e por quê; (3) estatística e p-valor; (4) **tamanho de efeito** em unidade de negócio (minutos, R$, %) e/ou padronizado; (5) decisão ao nível α = 5% e **frase de negócio**.

In [ ]:
ALFA = 0.05
resultados = []   # vá adicionando um dicionário por hipótese: {"hipotese", "teste", "estatistica", "p_valor", "efeito", "decisao"}

### Tarefa 9.1 — H1: chuva × tempo de entrega

In [ ]:
# Sua solução aqui


*Sua resposta aqui*

### Tarefa 9.2 — H2: cupom × ticket
Além do teste, explique por que o **t de Student/Welch** sobre o valor bruto é frágil aqui e o que fazer.

In [ ]:
# Sua solução aqui


*Sua resposta aqui*

### Tarefa 9.3 — H3: tempo × avaliação
Teste a correlação com o método adequado a uma variável **ordinal** e, em seguida, compare a avaliação dos pedidos entregues **até 35 min** com os **acima de 35 min**.

In [ ]:
# Sua solução aqui


*Sua resposta aqui*

### Tarefa 9.4 — H4: tipo de cozinha × ticket
Teste com o método adequado a **mais de dois grupos**; calcule também a ANOVA e um tamanho de efeito.

In [ ]:
# Sua solução aqui


*Sua resposta aqui*

### Tarefa 9.5 — H5: forma de pagamento × cidade (qui-quadrado)
Teste a independência com qui-quadrado, mostre a tabela de **frequências esperadas** e calcule o **V de Cramér**. Depois, como contraste, aplique o mesmo teste a `cupom × cliente_novo`.

In [ ]:
# Sua solução aqui


*Sua resposta aqui*

### Tarefa 9.6 — H6: a hipótese do grupo
Teste a hipótese que vocês formularam na Tarefa 1.2, com as mesmas 5 partes. Se possível, **controle uma variável confundidora**.

In [ ]:
# Sua solução aqui


*Sua resposta aqui*

### Tarefa 9.7 — Tabela-resumo das hipóteses
Transforme a lista `resultados` num DataFrame legível (p-valor formatado). É a tabela que vai para o README.

In [ ]:
# Sua solução aqui


---
## Etapa 10 — Conclusões para a diretoria

### Tarefa 10.1 — A figura principal
Construa **uma** figura (pode ter painéis) que resuma a história mais importante do projeto para a diretoria. Ela vai para o README e para a abertura da apresentação. Salve em `saidas/figura_principal.png` (dpi ≥ 150).

In [ ]:
# Sua solução aqui


### Tarefa 10.2 — Cinco achados
Escreva os **5 achados** mais importantes para a diretoria. Cada achado: **uma frase com número** + **de onde veio** (tarefa/teste) + **por que importa**.

*Sua resposta aqui*

### Tarefa 10.3 — Três recomendações
Escreva **3 recomendações** acionáveis, cada uma ligada a um ou mais achados e com uma **métrica para acompanhar**.

*Sua resposta aqui*

### Tarefa 10.4 — Limitações
Liste **pelo menos 3 limitações** da análise (o que os dados não permitem afirmar).

*Sua resposta aqui*

---
## Etapa 11 — Publicação

> Uma análise que só roda no seu computador não existe para a empresa. Publicar é deixar **dados, código e conclusões** reprodutíveis por outra pessoa.

### Tarefa 11.1 — Base limpa e dicionário
Salve a base tratada em `saidas/delivery_pedidos_limpo.csv` (sem colunas auxiliares de cálculo) e gere automaticamente um **dicionário de dados** da base limpa: coluna, tipo, % de faltantes e um exemplo de valor.

In [ ]:
# Sua solução aqui


### Tarefa 11.2 — README
Preencham o `modelo_README.md` desta pasta e salvem como `README.md` na raiz do repositório. Abaixo, escrevam o trecho **"Principais achados"** exatamente como ficará no README (com a figura principal referenciada).

*Sua resposta aqui*

### Tarefa 11.3 — Checklist de publicação
Antes de entregar, confiram (marquem com `x`):

- [ ] Repositório Git criado e repositório público.
- [ ] `README.md` preenchido a partir do `modelo_README.md`, com a figura principal aparecendo.
- [ ] O CSV original **não** foi alterado; a base limpa está em `saidas/`.
- [ ] *Kernel → Restart & Run All* roda sem erros do início ao fim.
- [ ] Nenhum gráfico sem título, rótulos e unidade.

> 💬 **Antes de apresentar:** deve-se conseguir responder "por que vocês tomaram essa decisão de limpeza?" e "o que esse p-valor significa?" sobre **qualquer** etapa.

---
## Resumo da metodologia

| Etapa | Pergunta que responde | Ferramentas |
|---|---|---|
| 1. Perguntas e hipóteses | o que queremos saber? | markdown, H0/H1 |
| 2. Carga e inspeção | o que veio e com que problemas? | `info`, `nunique`, `isna`, `describe`, `duplicated` |
| 3. Limpeza | como deixar os dados confiáveis? | `drop`, `to_datetime`, `.str`, `map`, `to_numeric`, `fillna` |
| 4. Outliers | é erro ou é real? | boxplot, z-score, IQR, investigação, flags |
| 5. Descritiva | como cada variável se comporta? | `describe`, CV, assimetria, percentis |
| 6. Sumarização e granularidade | o que muda por grupo e por período? | `groupby`/`agg`, `pivot_table`, `crosstab`, `resample` |
| 7. Visualização | como mostrar a resposta? | seaborn + estilização |
| 8. Correlação | o que anda junto? | `corr`, heatmap, scatter |
| 9. Hipóteses | é real ou acaso? e importa? | t de Welch, Mann-Whitney, Spearman, Kruskal-Wallis, χ², tamanho de efeito |
| 10. Conclusões | o que a diretoria deve fazer? | 5 achados + 3 recomendações + limitações |
| 11. Publicação | outra pessoa consegue reproduzir? | CSV limpo, README, Git |